In [1]:
# Ignore warnings for cleaner logs
import warnings
warnings.simplefilter("ignore")

In [2]:
!pip install -q sacrebleu wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.7 MB/s eta 0:00:00


In [3]:
# Imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import AutoTokenizer, T5ForConditionalGeneration
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import os
from kaggle_secrets import UserSecretsClient
from sklearn.model_selection import train_test_split
import sacrebleu
import wandb
import re
import math
from sacrebleu.metrics import CHRF
from pprint import pprint

2026-01-10 10:28:15.836353: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768040896.025316      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768040896.080882      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768040896.522306      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768040896.522363      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768040896.522366      24 computation_placer.cc:177] computation placer alr

In [4]:
# Comment out if you dont want WANDB logging
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb")
wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shreshth1110 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
# Training Configuration
class CONFIG():
    def __init__(self):
        self.MODEL_NAME  = "google/byt5-base"
        self.MAX_SOURCE_LEN = 512
        self.MAX_TARGET_LEN = 512
        self.SEED = 42
        self.TEST_SIZE = 0.1
        self.BIDIRECTIONAL_SPLIT = False
        self.SENTENCE_ALIGNMENT = False
        self.ASCII_NORM = True
        self.GAP_NORM = True
        self.BATCH_SIZE = 2
        self.EPOCHS = 10
        self.LR = 2e-4
        self.WEIGHT_DECAY = 0.01
        self.SAVE_PATH = "./byt5_base_dpc_v11"
        self.ES_PATIENCE = 3
        self.DECODER_BEAM_WIDTH = 4
        self.WANDB_EXPT_NAME = "byt5-base-v11"
        self.GRAD_ACC_STEPS = 2
        self.LABEL_SMOOTHING = 0.01
        self.REPETITION_PENALTY = 1.2
        self.DESC = "Low Epochs with LS=0.01"

cfg = CONFIG()

torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(cfg.SEED)
    torch.cuda.manual_seed_all(cfg.SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PREFIX = "translate Akkadian to English: "

In [6]:
# Comment out if you dont want WANDB logging
wandb.init(
    project="assyrian-mt",
    name=cfg.WANDB_EXPT_NAME,
    config=cfg
)

wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260110_102831-c20bfkog
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run byt5-base-v11
wandb: ⭐️ View project at https://wandb.ai/shreshth1110/assyrian-mt
wandb: 🚀 View run at https://wandb.ai/shreshth1110/assyrian-mt/runs/c20bfkog


In [7]:
print("Training Details: ")
pprint(cfg.__dict__)

Training Details: 
{'ASCII_NORM': True,
 'BATCH_SIZE': 2,
 'BIDIRECTIONAL_SPLIT': False,
 'DECODER_BEAM_WIDTH': 4,
 'DESC': 'Low Epochs with LS=0.01',
 'EPOCHS': 10,
 'ES_PATIENCE': 3,
 'GAP_NORM': True,
 'GRAD_ACC_STEPS': 2,
 'LABEL_SMOOTHING': 0.01,
 'LR': 0.0002,
 'MAX_SOURCE_LEN': 512,
 'MAX_TARGET_LEN': 512,
 'MODEL_NAME': 'google/byt5-base',
 'REPETITION_PENALTY': 1.2,
 'SAVE_PATH': './byt5_base_dpc_v11',
 'SEED': 42,
 'SENTENCE_ALIGNMENT': False,
 'TEST_SIZE': 0.1,
 'WANDB_EXPT_NAME': 'byt5-base-v11',
 'WEIGHT_DECAY': 0.01}


In [8]:
# Data
train = pd.read_csv('/kaggle/input/deep-past-initiative-machine-translation/train.csv')
sentences_oare = pd.read_csv('/kaggle/input/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv')

In [9]:
# Serntence Alignment
if cfg.SENTENCE_ALIGNMENT:
    df = train.merge(
        sentences_oare,
        left_on="oare_id",
        right_on="text_uuid",
        how="left",
        suffixes=("_train", "_sentcs")
    )
    
    # -----------------
    # Helpers
    # -------------------------
    chrf = CHRF(word_order=2)
    
    def split_translation_clauses(text):
        return [
            t.strip()
            for t in text.replace(";", ".").split(".")
            if t.strip()
        ]
    
    # -------------------------
    # Alignment logic
    # -------------------------
    aligned_pairs = []
    
    for oare_id, g in df.groupby("oare_id"):
    
        # tablet-level data (always available)
        full_translit = g.iloc[0]["transliteration"]
        full_translation = g.iloc[0]["translation_train"]
    
        # check if sentence-level alignment exists
        has_sentence_alignment = (
            g["first_word_number"].notna().any()
            and g["translation_sentcs"].notna().any()
        )
    
        # --------------------------------------------------
        # CASE 1: sentence-level alignment EXISTS → expand
        # --------------------------------------------------
        if has_sentence_alignment:
    
            g = g.dropna(subset=["first_word_number"]).sort_values("first_word_number")
    
            full_tokens = full_translit.split()
            translation_clauses = split_translation_clauses(full_translation)
    
            # sentence boundaries
            word_starts = g["first_word_number"].astype(int).tolist()
            word_starts.append(len(full_tokens) + 1)
    
            for idx, row in enumerate(g.itertuples()):
    
                start = int(row.first_word_number - 1)
                end = int(word_starts[idx + 1] - 1)
                translit_sentence = " ".join(full_tokens[start:end]).strip()
                anchor = str(row.translation_sentcs).strip()
    
                if not translit_sentence or not anchor:
                    continue
    
                # fuzzy match translation clauses
                scores = []
                for j, clause in enumerate(translation_clauses):
                    score = chrf.sentence_score(anchor, [clause]).score
                    scores.append((score, j))
    
                if not scores:
                    continue
    
                best_idx = max(scores)[1]
                candidate = translation_clauses[best_idx]
    
                # expansion heuristic
                if best_idx + 1 < len(translation_clauses):
                    expanded = candidate + ", " + translation_clauses[best_idx + 1]
                    if chrf.sentence_score(anchor, [expanded]).score > \
                       chrf.sentence_score(anchor, [candidate]).score:
                        candidate = expanded
    
                aligned_pairs.append({
                    "oare_id": oare_id,
                    "translit": translit_sentence,
                    "translation": candidate,
                    "alignment_type": "sentence",
                    "anchor": anchor
                })
    
        # --------------------------------------------------
        # CASE 2: NO sentence alignment → keep tablet-level
        # --------------------------------------------------
        else:
            aligned_pairs.append({
                "oare_id": oare_id,
                "translit": full_translit,
                "translation": full_translation,
                "alignment_type": "tablet",
                "anchor": None
            })
    # -------------------------
    # Save result
    # -------------------------
    train = pd.DataFrame(aligned_pairs)
    train['transliteration'] = train['translit']
    print("Total aligned pairs:", len(train))
    print(train["alignment_type"].value_counts())

In [10]:
# ASCII Normalization and Gap Normalization
ASCII_TO_DIACRITIC = {
    # Sibilants
    "sz": "š",
    "SZ": "Š",
    "Sz": "Š",
    "sh": "š",
    "SH": "Š",
    "Sh": "Š",
    # Emphatics (comma notation)
    "s,": "ṣ",
    "S,": "Ṣ",
    "t,": "ṭ",
    "T,": "Ṭ",
    "z,": "ẓ",
    "Z,": "Ẓ",
    # Emphatics (dot-under notation)
    ".s": "ṣ",
    ".S": "Ṣ",
    ".t": "ṭ",
    ".T": "Ṭ",
    ".z": "ẓ",
    ".Z": "Ẓ",
    # H variants
    "h,": "ḫ",
    "H,": "Ḫ",
    ".h": "ḫ",
    ".H": "Ḫ",
    "hh": "ḫ",
    "HH": "Ḫ",
    # Aleph
    "'": "ʾ",
    # Subscript numbers (common ASCII alternatives)
    "s2": "š",
    "S2": "Š",
    "s3": "ś",
    "S3": "Ś",
}

VOWEL_SUBSCRIPTS = {
    "a2": "á",
    "a3": "à",
    "e2": "é",
    "e3": "è",
    "i2": "í",
    "i3": "ì",
    "u2": "ú",
    "u3": "ù",
    "A2": "Á",
    "A3": "À",
    "E2": "É",
    "E3": "È",
    "I2": "Í",
    "I3": "Ì",
    "U2": "Ú",
    "U3": "Ù",
}


def normalize_ascii_to_diacritics(text: str) -> str:
    """Convert ASCII transliteration conventions to proper diacritics."""
    result = text
    for ascii_seq, diacritic in sorted(
        ASCII_TO_DIACRITIC.items(), key=lambda x: -len(x[0])
    ):
        result = result.replace(ascii_seq, diacritic)
    for ascii_seq, diacritic in VOWEL_SUBSCRIPTS.items():
        result = result.replace(ascii_seq, diacritic)
    return result


def normalize_gaps(text: str) -> str:
    """Normalize gap/damage markers: x→<gap>, x x x→<big_gap>."""
    tokens = text.split()
    result = []
    i = 0

    while i < len(tokens):
        token = tokens[i]
        if token.lower() == "x":
            x_count = 1
            while i + x_count < len(tokens) and tokens[i + x_count].lower() == "x":
                x_count += 1
            result.append("<gap>" if x_count == 1 else "<big_gap>")
            i += x_count
        else:
            if token.lower().startswith("x-"):
                result.append("<gap>" + token[1:])
            elif token.lower().endswith("-x"):
                result.append(token[:-1] + "-<gap>")
            else:
                result.append(token)
            i += 1

    text_result = " ".join(result)
    text_result = re.sub(r"<gap>\s+<big_gap>", "<big_gap>", text_result)
    text_result = re.sub(r"<big_gap>\s+<gap>", "<big_gap>", text_result)
    text_result = re.sub(r"(<gap>\s*){2,}", "<big_gap> ", text_result)
    return text_result.strip()


def apply_host_normalizations(text, ascii_norm_flag, gap_norm_flag):
    """Apply all host-recommended normalizations."""
    
    if ascii_norm_flag:
        text = normalize_ascii_to_diacritics(text)
    if gap_norm_flag:
        text = normalize_gaps(text)
    return text

In [11]:
# Pre-processing
train['transliteration'] = train['transliteration'].apply(lambda x: apply_host_normalizations(
    x, cfg.ASCII_NORM, cfg.GAP_NORM))

In [12]:
# Model and Tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(cfg.MODEL_NAME).to(DEVICE)

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [13]:
# NMT Dataset Class
class NMTDataset(Dataset):
    def __init__(self, df, tokenizer, max_source_len=512, max_target_len=256):
        self.df = df
        self.tokenizer = tokenizer
        self.max_source_len = max_source_len
        self.max_target_len = max_target_len

        self.sources = self.df["input_text"].tolist()
        self.targets = self.df["target_text"].tolist()

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        src = self.sources[idx]
        tgt = self.targets[idx]

        src_enc = self.tokenizer(
            src,
            truncation=True,
            padding="max_length",
            max_length=self.max_source_len,
            return_tensors="pt",
        )

        tgt_enc = self.tokenizer(
            tgt,
            truncation=True,
            padding="max_length",
            max_length=self.max_target_len,
            return_tensors="pt",
        )

        labels = tgt_enc["input_ids"]
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": src_enc["input_ids"].squeeze(0),
            "attention_mask": src_enc["attention_mask"].squeeze(0),
            "labels": labels.squeeze(0),
        }

In [14]:
# Train-Val Split and Dataloaders
train_idx, valid_idx = train_test_split(
    np.arange(len(train)),
    test_size=cfg.TEST_SIZE,            
    shuffle=True,                  
    random_state=42
)

train_df = train.iloc[train_idx].reset_index(drop=True)
valid_df = train.iloc[valid_idx].reset_index(drop=True)

def create_bidirectional_data(df):
    # Akkadian -> English
    df_fwd = df.copy()
    df_fwd['input_text'] = "translate Akkadian to English: " + df_fwd['transliteration'].astype(str)
    df_fwd['target_text'] = df_fwd['translation'].astype(str)
    
    # English -> Akkadian
    df_bwd = df.copy()
    df_bwd['input_text'] = "translate English to Akkadian: " + df_bwd['translation'].astype(str)
    df_bwd['target_text'] = df_bwd['transliteration'].astype(str)
    
    # Combined Data
    df_combined = pd.concat([df_fwd, df_bwd], ignore_index=True)
    df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return df_combined

def create_unidirectional_data(df):
    df['input_text'] = "translate Akkadian to English: " + df['transliteration'].astype(str)
    df['target_text'] = df['translation'].astype(str)
    return df

# Train Split
if cfg.BIDIRECTIONAL_SPLIT:
    train_df = create_bidirectional_data(train_df)
else:
    train_df = create_unidirectional_data(train_df)

# Validation Split
valid_df = create_unidirectional_data(valid_df)

print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(valid_df)}")

train_ds = NMTDataset(train_df, tokenizer, cfg.MAX_SOURCE_LEN, cfg.MAX_TARGET_LEN)
valid_ds = NMTDataset(valid_df, tokenizer, cfg.MAX_SOURCE_LEN, cfg.MAX_TARGET_LEN)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False
)

Train samples: 1404
Val samples:   157


In [15]:
# Loss, Optimizer, Scheduler
loss_fn = nn.CrossEntropyLoss(
    ignore_index=-100,
    label_smoothing=cfg.LABEL_SMOOTHING,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY, fused=True)

num_update_steps_per_epoch = math.ceil(len(train_loader) / cfg.GRAD_ACC_STEPS)
num_training_steps = num_update_steps_per_epoch * cfg.EPOCHS

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_training_steps,
    eta_min=0.0,
)

In [16]:
# Computing Grad Norm
def compute_grad_norm(model):
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    return total_norm ** 0.5

In [17]:
# Train one epoch
def train_epoch(model, loader, optimizer, scheduler, loss_fn, grad_accum_steps=4):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    last_grad_norm = 0.0

    for step, batch in enumerate(tqdm(loader)):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
            use_cache=False,
        )

        logits = outputs.logits
        labels = batch["labels"]

        loss = loss_fn(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
        )

        # scale loss for gradient accumulation
        loss = loss / grad_accum_steps
        loss.backward()

        # optimizer + scheduler step ONLY after accumulation
        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader):

            # Gradient clipping (HF default)
            last_grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item()

    return total_loss / len(loader), last_grad_norm

In [18]:
# Compute Val Loss
@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss = 0.0

    for batch in tqdm(loader, desc='Validation: ', total=len(loader)):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"], 
            use_cache=False,
        )

        logits = outputs.logits
        labels = batch["labels"]

        loss = loss_fn(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
        )

        total_loss += loss.item()

    return total_loss / len(loader)

In [19]:
# Compute Val Metrics with Inference Settings
@torch.no_grad()
def validate_generation(model, loader, tokenizer, max_len, num_beams):
    model.eval()

    preds, refs = [], []

    for batch in tqdm(loader, desc='Metrics: ', total=len(loader)):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"]

        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_len,
            num_beams=num_beams,
            length_penalty=1.0,
            early_stopping=True,
            repetition_penalty=cfg.REPETITION_PENALTY
        )

        pred_texts = tokenizer.batch_decode(
            generated_ids, skip_special_tokens=True
        )

        labels = labels.clone()
        labels[labels == -100] = tokenizer.pad_token_id

        ref_texts = tokenizer.batch_decode(
            labels,
            skip_special_tokens=True,
        )

        preds.extend(pred_texts)
        refs.extend(ref_texts)

    bleu = sacrebleu.corpus_bleu(
        preds,
        [refs],
        tokenize="13a",
        lowercase=False,
    ).score

    chrf = sacrebleu.corpus_chrf(
        preds,
        [refs],
        word_order=2   # chrF++
    ).score

    geo_mean = (bleu * chrf) ** 0.5

    return bleu, chrf, geo_mean
    {
        "BLEU": bleu,
        "chrF++": chrf,
        "GEO_MEAN": geo_mean,
    }

In [20]:
# Training Loop
best_loss = float("inf")
early_stop_counter = 0

print("Training started")
for epoch in range(cfg.EPOCHS):
    current_lr = optimizer.param_groups[0]["lr"]
    train_loss, grad_norm = train_epoch(model, train_loader, optimizer, 
                                        scheduler, loss_fn, cfg.GRAD_ACC_STEPS)
    
    val_loss = validate(model, valid_loader)
    # scheduler.step(val_loss)
    
    print(
        f"Epoch {epoch:02d} | "
        f"Training Loss: {train_loss:.4f} | "
        f"Grad Norm: {grad_norm:.4f} | "
        f"Val Loss: {val_loss:.3f} | "
        f"LR: {current_lr:.2e}"
    )

    # Comment out if you dont want WANDB logging
    wandb.log(
        {
            "Train Loss": train_loss,
            "Grad Norm": grad_norm,
            "Val Loss": val_loss,
            "learning_rate": current_lr
        }
    )
    # Till here

    # ===== Early stopping =====
    if val_loss < best_loss:
        best_loss = val_loss
        early_stop_counter = 0
        model.save_pretrained(cfg.SAVE_PATH)
        tokenizer.save_pretrained(cfg.SAVE_PATH)
        print("Best Model Saved!")
    else:
        early_stop_counter += 1

    if early_stop_counter >= cfg.ES_PATIENCE:
        print("Early stopping triggered!")
        break

Training started


  0%|          | 0/702 [00:00<?, ?it/s]

Validation:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch 00 | Training Loss: 0.5552 | Grad Norm: 0.6839 | Val Loss: 0.762 | LR: 2.00e-04
Best Model Saved!


  0%|          | 0/702 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch 08 | Training Loss: 0.2177 | Grad Norm: 0.7509 | Val Loss: 0.516 | LR: 1.91e-05


  0%|          | 0/702 [00:00<?, ?it/s]

Validation:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch 09 | Training Loss: 0.2167 | Grad Norm: 0.7871 | Val Loss: 0.515 | LR: 4.89e-06


In [21]:
# Loading best model
print("Loading best model")
tokenizer = AutoTokenizer.from_pretrained(cfg.SAVE_PATH)
model = T5ForConditionalGeneration.from_pretrained(cfg.SAVE_PATH).to(DEVICE);

Loading best model


In [22]:
# Generation Metrics on Val
print("Generation Metrics")
bleu, chrf, geo_mean = validate_generation(model, valid_loader, tokenizer, 
                                               cfg.MAX_TARGET_LEN, cfg.DECODER_BEAM_WIDTH)

print(
    f"BLEU: {bleu:.4f} | "
    f"CHRF++: {chrf:.4f} | "
    f"Metric: {geo_mean:.4f} | "
)

Generation Metrics


Metrics:   0%|          | 0/79 [00:00<?, ?it/s]

BLEU: 32.8426 | CHRF++: 52.4869 | Metric: 41.5187 | 
